# NLP Masterclass 06: HuggingFace Pipelines & Transfer Learning

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 05 (Pre-training & BERT)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"HuggingFace has 500K+ models on its Hub. You need a model for detecting toxic comments in production. How do you choose between fine-tuning a base model yourself vs. using an existing fine-tuned model? What are the risks of each approach?"*

---

## Why This Notebook Completes the NLP Track

NLP 01-05 taught you the THEORY: tokenization → embeddings → RNNs → attention → pre-training.

This notebook teaches the PRACTICE: how to use HuggingFace's ecosystem to solve real NLP tasks in production. Everything here directly connects to the main curriculum's NB11 (fine-tuning) and NB24-34 (LLMOps).

## Table of Contents
1. HuggingFace Hub: Finding & Evaluating Models
2. Pipelines: Zero-Code NLP
3. NLP Task Taxonomy (Classification, NER, QA, Summarization)
4. Fine-tuning with Trainer API
5. Production Deployment Patterns

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors pick the highest-downloaded model. Seniors evaluate: license compatibility, benchmark scores on THEIR domain, model size vs. latency budget, last commit date, and whether the model was trained on data similar to their production data.

**Mental Model:** The HuggingFace Hub is like npm/pip for ML models. But unlike software packages, ML models have SILENT failures — a bad model won't crash, it'll just give wrong answers confidently. You must evaluate before deploying.

**Common Junior Pitfall:** Using `pipeline()` in production without controlling batch size, device placement, or model quantization. Default pipeline settings are for demos, not production throughput.

---

In [ ]:
!pip install -q transformers datasets evaluate accelerate

## 1 · HuggingFace Hub: Model Selection Checklist

Before using ANY model, check:

| Criterion | Why It Matters | Where to Check |
|-----------|---------------|----------------|
| **License** | Apache-2.0 = commercial OK, CC-BY-NC = non-commercial only | Model card |
| **Downloads** | Popularity ≈ community trust | Hub page |
| **Benchmarks** | Does it perform on YOUR task? | Model card / Papers With Code |
| **Model size** | Fits your GPU/latency budget? | Model card |
| **Last updated** | Stale models may have unpatched issues | Hub page |
| **Training data** | Was it trained on data similar to yours? | Model card |
| **Base model** | What was it fine-tuned from? | Model card |

---
## 2 · Pipelines: Every NLP Task in One Line

HuggingFace `pipeline()` wraps tokenization + model + post-processing into one call.

In [ ]:
from transformers import pipeline

# ============================
# TASK 1: Sentiment Analysis
# ============================
print('=== Sentiment Analysis ===')
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

reviews = [
    'This product exceeded all my expectations!',
    'Worst purchase I have ever made. Total garbage.',
    'It works fine, nothing extraordinary though.',
]
results = classifier(reviews)
for review, result in zip(reviews, results):
    print(f'  [{result["label"]:>8} {result["score"]:.2%}] "{review[:50]}"')

# ============================
# TASK 2: Named Entity Recognition (NER)
# ============================
print('\n=== Named Entity Recognition ===')
ner = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')

text = 'Elon Musk announced that Tesla will open a new factory in Berlin, Germany next year.'
entities = ner(text)
print(f'  Text: "{text}"')
for ent in entities:
    print(f'  {ent["entity_group"]:>5}: "{ent["word"]}" (confidence: {ent["score"]:.2%})')

In [ ]:
from transformers import pipeline

# ============================
# TASK 3: Question Answering (Extractive)
# ============================
print('=== Extractive Question Answering ===')
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context = """PyTorch was developed by Meta's AI Research lab (FAIR). It was first released 
in September 2016 and has since become the most popular deep learning framework for research.
As of 2024, it powers over 60% of academic publications in machine learning."""

questions = [
    'Who developed PyTorch?',
    'When was PyTorch first released?',
    'What percentage of academic publications use PyTorch?',
]

for q in questions:
    result = qa(question=q, context=context)
    print(f'  Q: {q}')
    print(f'  A: "{result["answer"]}" (confidence: {result["score"]:.2%})')
    print()

# ============================
# TASK 4: Summarization
# ============================
print('=== Summarization ===')
summarizer = pipeline('summarization', model='facebook/bart-large-cnn')

article = """The field of artificial intelligence has seen remarkable progress in recent years, 
particularly with the development of large language models. These models, trained on vast amounts 
of text data, have demonstrated capabilities in understanding and generating human language that 
were previously thought impossible. Companies like OpenAI, Google, and Meta have all released 
increasingly powerful models, each pushing the boundaries of what AI can achieve. However, this 
rapid progress has also raised concerns about safety, bias, and the potential misuse of these 
powerful tools. Researchers are now working on alignment techniques to ensure these models 
behave in ways that are helpful, honest, and harmless."""

summary = summarizer(article, max_length=60, min_length=20)
print(f'  Original: {len(article.split())} words')
print(f'  Summary:  "{summary[0]["summary_text"]}"')
print(f'  Compressed to: {len(summary[0]["summary_text"].split())} words')

In [ ]:
from transformers import pipeline

# ============================
# TASK 5: Zero-Shot Classification
# ============================
print('=== Zero-Shot Classification ===')
print('(No training data needed! Classify into ANY categories)')

zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

text = 'The patient showed elevated troponin levels and ST-segment depression on the ECG.'
labels = ['cardiology', 'neurology', 'orthopedics', 'dermatology']

result = zero_shot(text, labels)
print(f'  Text: "{text[:60]}..."')
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f'    {label:>15}: {bar} {score:.2%}')

# Production use case: route support tickets without training
print('\n--- Support Ticket Routing ---')
ticket = 'My card was charged twice for the same order and I want a refund.'
departments = ['billing', 'technical support', 'shipping', 'account management']
result = zero_shot(ticket, departments)
print(f'  Ticket: "{ticket[:60]}"')
print(f'  Route to: {result["labels"][0]} ({result["scores"][0]:.2%})')

---
## 3 · Fine-tuning with Trainer API

When zero-shot or existing models aren't accurate enough, fine-tune:

```
Pre-trained Model (knows language)  +  Your Data (100-10K labels)
         │                                        │
         └──────────── Trainer API ───────────────┘
                           │
                 Fine-tuned Model (knows your task)
```

This is the same `Trainer` used in NB11.  
For large models (7B+), use LoRA/QLoRA (NB11) instead of full fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np

# Create synthetic training data (in practice: use real labeled data)
train_texts = [
    'This is absolutely wonderful!', 'I love this product so much!',
    'Best experience ever, highly recommend!', 'Amazing quality and fast shipping!',
    'Terrible, complete waste of money.', 'Horrible customer service experience.',
    'Broke after one day, very disappointing.', 'Do not buy this, worst ever.',
] * 10  # Repeat for minimum training size
train_labels = [1,1,1,1,0,0,0,0] * 10

# Load model and tokenizer
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding=True, max_length=64)

dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
tokenized = dataset.map(tokenize_fn, batched=True)

# Training arguments
args = TrainingArguments(
    output_dir='./sentiment_model',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,      # Standard for BERT fine-tuning
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='no',       # Don't save checkpoints for demo
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

print('Fine-tuning DistilBERT for sentiment classification...')
print(f'  Base model: {model_name} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)')
print(f'  Training samples: {len(train_texts)}')
print(f'  Learning rate: 2e-5 (much lower than training from scratch)')

trainer.train()

# Test
model.eval()
test_texts = ['This is the best thing ever!', 'Absolutely terrible, I hate it.']
inputs = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    logits = model(**inputs).logits
    preds = torch.softmax(logits, dim=-1)

print(f'\nAfter fine-tuning:')
for text, pred in zip(test_texts, preds):
    label = 'Positive' if pred[1] > pred[0] else 'Negative'
    print(f'  "{text}" → {label} ({pred.max():.2%})')

---
## 4 · Production Deployment Patterns

| Pattern | Latency | Cost | Best For |
|---------|---------|------|----------|
| **API (OpenAI, HF Inference)** | 200-500ms | Pay per request | Prototyping, low volume |
| **Self-hosted (FastAPI + GPU)** | 50-100ms | Fixed GPU cost | Medium volume, custom models |
| **Optimized (ONNX/TensorRT)** | 5-20ms | Optimized GPU | High volume, latency-critical |
| **Edge (ONNX Runtime)** | 10-50ms | No GPU needed | On-device, privacy-sensitive |

### Connection to the Main Curriculum

| This Notebook | Main Track |
|--------------|------------|
| Pipeline basics | → NB15 (FastAPI serving) |
| Fine-tuning with Trainer | → NB11 (LoRA fine-tuning) |
| Zero-shot classification | → NB24 (Prompt engineering) |
| NER pipeline | → NB28 (Entity extraction for RAG) |
| Model selection | → NB11 (Model selection helper) |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How do you choose between fine-tuning a base model vs. using an existing fine-tuned model?*

**A:** Decision tree:
1. Is there an existing model fine-tuned for your EXACT task? → Try it first (free)
2. Does it meet your accuracy bar (>90% on your test set)? → Deploy it
3. If not, is your domain very specialized (legal, medical, finance)? → Fine-tune yourself
4. Do you have enough labeled data (>500 examples)? → Fine-tune
5. Too little data? → Use zero-shot classification or few-shot prompting (NB24)

**Risk of using existing models:** They may have been trained on data that doesn't match your distribution. A sentiment model trained on movie reviews may fail on medical patient feedback.

**Risk of fine-tuning yourself:** Overfitting on small datasets, catastrophic forgetting of pre-trained knowledge, and ongoing maintenance burden.

---

## ✅ Knowledge Check

### Q1: Pipeline internals
What three steps does `pipeline('sentiment-analysis')(text)` perform internally?

<details><summary>👉 Answer</summary>

1. **Tokenization**: Convert text to token IDs using the model's tokenizer
2. **Model inference**: Feed token IDs through the model to get logits
3. **Post-processing**: Apply softmax to logits, map to labels (POSITIVE/NEGATIVE), return structured output
</details>

### Q2: Learning rate
Why is the fine-tuning learning rate (2e-5) SO much lower than training from scratch (~1e-3)?

<details><summary>👉 Answer</summary>

The pre-trained weights already encode useful language knowledge. A high learning rate would destroy these learned features (catastrophic forgetting). A low learning rate makes small, careful adjustments to adapt the existing knowledge to the new task. This is like adjusting a pre-tuned piano rather than building one from scratch.
</details>

### Q3: Zero-shot vs fine-tuning
You have 50 labeled examples and need 95% accuracy. Should you fine-tune or use zero-shot?

<details><summary>👉 Answer</summary>

Start with zero-shot classification — 50 examples is often too few for reliable fine-tuning. If zero-shot doesn't hit 95%, try few-shot prompting with an LLM (NB24). If that doesn't work either, invest in labeling more data (aim for 500+) before fine-tuning. 50 examples should be your TEST set, not training set.
</details>

---

## 🔨 Practical Practice

### Exercise 1: NER Pipeline for RAG
Build a pipeline that extracts named entities from a document, then creates metadata tags for each entity. This is preprocessing for RAG (NB28).

### Exercise 2: Model Comparison
Compare 3 sentiment models on the same 20 test sentences: `distilbert-base-uncased-finetuned-sst-2-english`, `nlptown/bert-base-multilingual-uncased-sentiment`, and a zero-shot classifier. Which is most accurate? Which is fastest?

### Exercise 3: Batch Inference Optimization
Time the pipeline with batch_size=1 vs batch_size=32 on 1000 sentences. Calculate the speedup. Then try using `model.half()` (FP16) and measure speedup and accuracy change.

---

## 🎉 NLP Track Complete!

You've covered the full evolution:

```
NLP 01: Text → Tokens          (how computers read)
NLP 02: Tokens → Vectors        (how meaning becomes geometry)
NLP 03: Vectors → Sequences     (how RNNs process language)
NLP 04: Sequences → Attention   (how context is captured)
NLP 05: Attention → Pre-training (how models learn language)
NLP 06: Pre-training → Production (how to ship it)
```

**Next →** Return to the main track: NB11 (LoRA Fine-tuning) or NB24 (Prompt Engineering)